# Laboratorio 4 — Agentes Inteligentes con CrewAI
## Fact-Checker de Patrimonio Cultural Boliviano

**Módulo 10:** NLP, Text Analytics, Search — MIAV1E3  
**Docente:** Danny Huanca Sevilla  
**Verónica Tancara**

---

### Descripción del sistema

Se implementa un sistema multi-agente que verifica automáticamente claims sobre el origen de danzas
y expresiones culturales bolivianas. Ante la creciente atribución errónea de danzas bolivianas a otros
países (particularmente en redes sociales), este sistema busca evidencia verificable para emitir un
veredicto fundamentado.

**Agentes:**
1. **Investigador Cultural** — Busca fuentes históricas, declaratorias oficiales y artículos sobre el origen de la danza
2. **Verificador de Hechos** — Contrasta fuentes, identifica evidencia a favor y en contra del claim
3. **Redactor de Veredicto** — Emite un veredicto estructurado con evidencia y fuentes

**Stack tecnológico:**
- **CrewAI** — Orquestación multi-agente
- **Google Gemini** (gemini-2.5-flash) — LLM
- **Serper API** — Búsqueda web en tiempo real

## 1. Configuración del entorno

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("GOOGLE_API_KEY"), "Falta GOOGLE_API_KEY en .env"
assert os.getenv("SERPER_API_KEY"), "Falta SERPER_API_KEY en .env"
print("API keys cargadas correctamente.")

API keys cargadas correctamente.


In [2]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

# LLM: Google Gemini
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

class SerperSearchTool(BaseTool):
    name: str = "Buscar en internet"
    description: str = (
        "Busca información en internet sobre un tema. "
        "Recibe un query de búsqueda y devuelve resultados con título, snippet y URL."
    )

    def _run(self, query: str) -> str:
        response = requests.post(
            "https://google.serper.dev/search",
            json={"q": query, "gl": "bo", "hl": "es", "num": 10},
            headers={"X-API-KEY": os.getenv("SERPER_API_KEY")},
        )
        data = response.json()
        results = []
        for r in data.get("organic", []):
            results.append(
                f"- Título: {r['title']}\n"
                f"  Snippet: {r.get('snippet', 'Sin resumen')}\n"
                f"  URL: {r.get('link', 'Sin URL')}"
            )
        return "\n\n".join(results) if results else "No se encontraron resultados."

search_tool = SerperSearchTool()
print(f"LLM: {llm.model}")
print(f"Herramienta: {search_tool.name}")

LLM: gemini-2.5-flash
Herramienta: Buscar en internet


## 2. Definición de agentes

In [3]:
# --- Agente 1: Investigador Cultural ---
investigador = Agent(
    role="Investigador de Patrimonio Cultural",
    goal="Buscar evidencia sobre el origen de la danza mencionada en el claim.",
    backstory=(
        "Eres un investigador cultural especializado en folklore andino. "
        "Usa tu herramienta de búsqueda para encontrar evidencia sobre el origen "
        "de la danza. Busca fuentes de Bolivia y Perú."
    ),
    tools=[search_tool],
    llm=llm,
    verbose=True,
)

# --- Agente 2: Verificador de Hechos ---
verificador = Agent(
    role="Verificador de Hechos Culturales",
    goal="Contrastar las fuentes encontradas y emitir evaluación preliminar.",
    backstory=(
        "Eres un fact-checker profesional. Evalúa las fuentes del investigador, "
        "identifica el consenso y emite una evaluación concisa."
    ),
    llm=llm,
    verbose=True,
)

# --- Agente 3: Redactor de Veredicto ---
redactor = Agent(
    role="Redactor de Veredicto de Fact-Check",
    goal="Emitir veredicto claro y fundamentado sobre el claim.",
    backstory=(
        "Eres editor de una agencia de fact-checking. Emite veredicto usando: "
        "VERDADERO, PARCIALMENTE CIERTO, FALSO, o SIN EVIDENCIA SUFICIENTE. "
        "Incluye evidencia clave y fuentes."
    ),
    llm=llm,
    verbose=True,
)

print("Agentes creados:")
for a in [investigador, verificador, redactor]:
    print(f"  • {a.role}")

Agentes creados:
  • Investigador de Patrimonio Cultural
  • Verificador de Hechos Culturales
  • Redactor de Veredicto de Fact-Check


## 3. Claims a verificar

Se definen las aseveraciones (claims) que circulan frecuentemente en redes sociales, particularmente en TikTok, donde creadores de contenido peruanos atribuyen danzas bolivianas a Perú.

In [4]:
claims = [
    "La Morenada es una danza originaria de Perú",
    "La Diablada se originó en Puno, Perú",
    "El Caporal es una danza peruana",
    "El Pujllay es una danza boliviana",
]

print(f"Claims a verificar: {len(claims)}")
for i, c in enumerate(claims, 1):
    print(f"  {i}. \"{c}\"")

Claims a verificar: 4
  1. "La Morenada es una danza originaria de Perú"
  2. "La Diablada se originó en Puno, Perú"
  3. "El Caporal es una danza peruana"
  4. "El Pujllay es una danza boliviana"


## 4. Definición de tareas y ejecución

Se define una función que crea el crew para cada claim y lo ejecuta. Cada claim pasa por los tres agentes secuencialmente.

In [5]:
from IPython.display import Markdown, display

def verificar_claim(claim: str) -> dict:
    """Ejecuta el crew de fact-checking para un claim dado."""

    tarea_investigacion = Task(
        description=(
            f'Busca en internet sobre: "{claim}". '
            "Usa UNA búsqueda con query: nombre de la danza + 'origen Bolivia Perú'. "
            "Lista las fuentes encontradas con título, URL y si apoya o contradice el claim."
        ),
        expected_output="Lista de fuentes con título, URL y posición respecto al claim.",
        agent=investigador,
    )

    tarea_verificacion = Task(
        description=(
            f'Con las fuentes encontradas, verifica: "{claim}". '
            "Evalúa credibilidad de fuentes y emite evaluación preliminar."
        ),
        expected_output="Evaluación con evidencia a favor y en contra del claim.",
        agent=verificador,
    )

    tarea_veredicto = Task(
        description=(
            f'Emite veredicto final para: "{claim}". Formato:\n'
            "1. CLAIM\n2. VEREDICTO (VERDADERO/FALSO/PARCIALMENTE CIERTO)\n"
            "3. RESUMEN (2 oraciones)\n4. EVIDENCIA CLAVE (3 puntos)\n"
            "5. FUENTES (URLs)\n6. CONTEXTO (1 párrafo)"
        ),
        expected_output="Veredicto estructurado con las 6 secciones.",
        agent=redactor,
    )

    crew = Crew(
        agents=[investigador, verificador, redactor],
        tasks=[tarea_investigacion, tarea_verificacion, tarea_veredicto],
        process=Process.sequential,
        verbose=True,
    )

    resultado = crew.kickoff()
    return {
        "claim": claim,
        "veredicto": resultado.raw,
        "tokens": resultado.token_usage,
    }

print("Función verificar_claim() definida.")

Función verificar_claim() definida.


## 5. Ejecución: verificación de todos los claims

In [6]:
resultados = []

for i, claim in enumerate(claims, 1):
    print(f"\n{'='*70}")
    print(f"  CLAIM {i}/{len(claims)}: \"{claim}\"")
    print(f"{'='*70}\n")

    resultado = verificar_claim(claim)
    resultados.append(resultado)

    display(Markdown(f"---\n### Veredicto — Claim {i}\n\n{resultado['veredicto']}"))

print(f"\n{'='*70}")
print(f"  Verificación completada: {len(resultados)} claims procesados")
print(f"{'='*70}")


  CLAIM 1/4: "La Morenada es una danza originaria de Perú"



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 903365b1-60d9-41e0-8c08-10b78794de8a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca en internet sobre: "La Morenada es una danza originaria de Perú". Usa UNA búsqueda con query:      │
│  nombre de la danza + 'origen Bolivia Perú'. Lista las fuentes encontradas con título, URL y si apoya o         │
│  contradice el claim.                                                                                           │
│  ID: 660392b2-110d-4772-98ec-798426c121e4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de Patrimonio Cultural                                                                     │
│                                                                                                                 │
│  Task: Busca en internet sobre: "La Morenada es una danza originaria de Perú". Usa UNA búsqueda con query:      │
│  nombre de la danza + 'origen Bolivia Perú'. Lista las fuentes encontradas con título, URL y si apoya o         │
│  contradice el claim.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: buscar_en_internet                                                                                       │
│  Args: {'query': 'La Morenada origen Bolivia Perú'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Emite veredicto final para: "El Pujllay es una danza boliviana". Formato:                                │
│  1. CLAIM                                                                                                       │
│  2. VEREDICTO (VERDADERO/FALSO/PARCIALMENTE CIERTO)                                                             │
│  3. RESUMEN (2 oraciones)                                                                                       │
│  4. EVIDENCIA CLAVE (3 puntos)                                                                                  │
│  5. FUENTES (URLs)                                                                                              │
│  6. CONTEXTO (1 párrafo)                                                                                        │
│  Agent: Redactor de Veredicto de Fact-Check                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b4a4889d-f611-4005-b11d-95204db696ca                                                                       │
│  Final Output: 1.  CLAIM                                                                                        │
│      El Pujllay es una danza boliviana.                                                                         │
│                                                                                                                 │
│  2.  VEREDICTO                                                                                                  │
│      VERDADERO                                                                                                  │
│                                                                                                                 │
│  3.  RESUMEN                                                                                                    │
│      El Pujllay es una danza ritual originaria de Bolivia, específicamente de la cultura Yampara de la región   │
│  de Chuquisaca. Ha sido reconocida por la UNESCO como Patrimonio Cultural Inmaterial de la Humanidad,           │
│  confirmando su profundo arraigo en el patrimonio boliviano.                                                    │
│                                                                                                                 │
│  4.  EVIDENCIA CLAVE                                                                                            │
│      *   La UNESCO ha inscrito "El Pujllay y el Ayarichi" como Patrimonio Cultural Inmaterial de la Humanidad,  │
│  describiéndolo como una manifestación cultural de la "cultura yampara", que es originaria de Bolivia           │
│  (departamento de Chuquisaca).                                                                                  │
│      *   Fuentes institucionales como Wikipedia y el Parlamento Andino describen explícitamente el Pujllay      │
│  como una "danza ritual boliviana" y lo asocian directamente con Tarabuco, una localidad en Bolivia.            │
│      *   Prácticamente todas las fuentes consultadas, incluyendo sitios culturales bolivianos y publicaciones   │
│  en redes sociales con enfoque boliviano, identifican consistentemente el Pujllay como una danza propia de      │
│  Bolivia, sin que exista evidencia que contradiga o ponga en duda este origen.                                  │
│                                                                                                                 │
│  5.  FUENTES                                                                                                    │
│      *   https://ich.unesco.org/es/RL/el-pujllay-y-el-ayarichi-musicas-y-danzas-de-la-cultura-yampara-00630     │
│      *   https://es.wikipedia.org/wiki/Pujllay_(danza)                                                          │
│      *                                                                                                          │
│  https://www.parlamentoandino.org/index.php/actualidad/noticias/125-pujllay-y-el-misticismo-de-tarabuco         │
│      *                                                                                                          │
│  https://www.facebook.com/ervcbolivia/posts/el-pujllay-es-una-danza-ritual-boliviana-de-la-cultura-yampara-ori  │
│  ginaria-de-la-/1487576750043628/                                                                               │
│      *                                                

---
### Veredicto — Claim 4

1.  CLAIM
    El Pujllay es una danza boliviana.

2.  VEREDICTO
    VERDADERO

3.  RESUMEN
    El Pujllay es una danza ritual originaria de Bolivia, específicamente de la cultura Yampara de la región de Chuquisaca. Ha sido reconocida por la UNESCO como Patrimonio Cultural Inmaterial de la Humanidad, confirmando su profundo arraigo en el patrimonio boliviano.

4.  EVIDENCIA CLAVE
    *   La UNESCO ha inscrito "El Pujllay y el Ayarichi" como Patrimonio Cultural Inmaterial de la Humanidad, describiéndolo como una manifestación cultural de la "cultura yampara", que es originaria de Bolivia (departamento de Chuquisaca).
    *   Fuentes institucionales como Wikipedia y el Parlamento Andino describen explícitamente el Pujllay como una "danza ritual boliviana" y lo asocian directamente con Tarabuco, una localidad en Bolivia.
    *   Prácticamente todas las fuentes consultadas, incluyendo sitios culturales bolivianos y publicaciones en redes sociales con enfoque boliviano, identifican consistentemente el Pujllay como una danza propia de Bolivia, sin que exista evidencia que contradiga o ponga en duda este origen.

5.  FUENTES
    *   https://ich.unesco.org/es/RL/el-pujllay-y-el-ayarichi-musicas-y-danzas-de-la-cultura-yampara-00630
    *   https://es.wikipedia.org/wiki/Pujllay_(danza)
    *   https://www.parlamentoandino.org/index.php/actualidad/noticias/125-pujllay-y-el-misticismo-de-tarabuco
    *   https://www.facebook.com/ervcbolivia/posts/el-pujllay-es-una-danza-ritual-boliviana-de-la-cultura-yampara-originaria-de-la-/1487576750043628/
    *   https://ruta1825.com.bo/ruta-1825/turismo/altiplano/pujllay-del-bicentenario-tradicion-historia-y-gratitud-a-la-pachamama
    *   https://es.scribd.com/document/350282529/Origenes-Del-Pujllay

6.  CONTEXTO
    El Pujllay es una danza ritual que se celebra anualmente en la región de Chuquisaca, Bolivia, especialmente en la localidad de Tarabuco, para conmemorar la Batalla de Cinti (también conocida como Batalla de Jumbate) ocurrida en 1816, y para agradecer a la Pachamama (Madre Tierra) por las cosechas. Es una expresión cultural vibrante de la etnia Yampara, caracterizada por sus trajes coloridos, sus imponentes monteras (sombreros) y el "pujllay" o "juego" de los hombres que danzan con espuelas, mientras las mujeres realizan movimientos más sutiles. Su reconocimiento por la UNESCO subraya su valor histórico y cultural para la humanidad, consolidando su identidad como parte intrínseca del patrimonio boliviano.


  Verificación completada: 4 claims procesados


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 6. Tabla resumen de veredictos

In [7]:
import re

def extraer_veredicto(texto: str) -> str:
    """Extrae el veredicto del texto generado."""
    for label in ["FALSO", "PARCIALMENTE CIERTO", "VERDADERO", "SIN EVIDENCIA SUFICIENTE"]:
        if label in texto.upper():
            return label
    return "NO DETERMINADO"

tabla = "| # | Claim | Veredicto |\n|---|-------|----------|\n"
for i, r in enumerate(resultados, 1):
    veredicto = extraer_veredicto(r["veredicto"])
    tabla += f"| {i} | {r['claim']} | **{veredicto}** |\n"

display(Markdown(tabla))

| # | Claim | Veredicto |
|---|-------|----------|
| 1 | La Morenada es una danza originaria de Perú | **FALSO** |
| 2 | La Diablada se originó en Puno, Perú | **FALSO** |
| 3 | El Caporal es una danza peruana | **FALSO** |
| 4 | El Pujllay es una danza boliviana | **VERDADERO** |


## 7. Métricas de ejecución

In [8]:
total_prompt = sum(r["tokens"].prompt_tokens for r in resultados)
total_completion = sum(r["tokens"].completion_tokens for r in resultados)
total_tokens = sum(r["tokens"].total_tokens for r in resultados)
total_requests = sum(r["tokens"].successful_requests for r in resultados)

print(f"Métricas totales ({len(resultados)} claims verificados):")
print(f"  Tokens de entrada:  {total_prompt:,}")
print(f"  Tokens de salida:   {total_completion:,}")
print(f"  Total de tokens:    {total_tokens:,}")
print(f"  Llamadas al LLM:    {total_requests}")
print(f"  Promedio por claim:  {total_tokens // len(resultados):,} tokens")

Métricas totales (4 claims verificados):
  Tokens de entrada:  447,876
  Tokens de salida:   90,939
  Total de tokens:    538,815
  Llamadas al LLM:    120
  Promedio por claim:  134,703 tokens
